# KenProbe v0 — Qwen 4B + Unsloth LoRA

Trains KenProbe as a citation-first research model: search decision, tool calls, source reading, verification, and cited answers.

Default config is the real v0 run: **1000 steps / 5000 HotpotQA samples / 2048 seq length**.


In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
%%capture
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
!pip install -U datasets trl accelerate bitsandbytes sentencepiece protobuf huggingface_hub pandas

In [ ]:
import os, json, random, shutil
from datetime import datetime, timezone

import torch
from datasets import Dataset, load_dataset
from huggingface_hub import login, HfApi

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## HF login

Run `login()` before uploading artifacts. Local download later uses `hf`, not `huggingface-cli`:

```powershell
hf auth login
hf download amaansyed27/kenprobe-adapters kenprobe-qwen35-4b-lora-step1000.zip --local-dir runs
```


In [ ]:
# login()

## Config

For smoke testing, change `MAX_STEPS` to 80 and `TRAIN_HOTPOT_SAMPLES` to 500. For real training, keep the defaults.


In [ ]:
BASE_MODEL = 'unsloth/Qwen3.5-4B'
# Fallback only if needed:
# BASE_MODEL = 'unsloth/Qwen3-4B-Instruct-2507'

PROJECT_NAME = 'kenprobe'
MODEL_SLUG = 'qwen35-4b'
HF_ADAPTER_REPO = 'amaansyed27/kenprobe-adapters'
HF_GGUF_REPO = 'amaansyed27/kenprobe-gguf'

MAX_SEQ_LENGTH = 2048
MAX_STEPS = 1000
TRAIN_HOTPOT_SAMPLES = 5000

EVAL_SIZE = 0.05
SEED = 3407
LOAD_IN_4BIT = False
LOAD_IN_16BIT = True

RUN_NAME = f'{PROJECT_NAME}-{MODEL_SLUG}-lora-step{MAX_STEPS}'
print('Run:', RUN_NAME)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_16bit=LOAD_IN_16BIT,
    fast_inference=False,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

print('Model and LoRA adapter loaded.')

## Behavior helpers

Uses explicit text-only ChatML to avoid Qwen3.5 processor/image-template inference errors.


In [ ]:
SYSTEM_PROMPT = """You are KenProbe, a citation-first research chatbot.

Rules:
1. For factual, current, benchmark, product, legal, medical, financial, or research questions, request search/tool use before answering.
2. Use structured tool calls in this exact XML-like wrapper:
   <tool_call>{\"name\":\"web_search\",\"arguments\":{\"query\":\"...\"}}</tool_call>
3. After sources are provided, answer only from the sources.
4. Cite source IDs inline as [S1], [S2].
5. If sources disagree, explain the disagreement.
6. If the evidence is insufficient, say \"I do not have enough evidence\" and explain what is missing.
7. Be concise, accurate, and avoid unsupported claims.
"""

text_tokenizer = getattr(tokenizer, 'tokenizer', tokenizer)

def to_chatml(messages, add_generation_prompt=False):
    parts = []
    for m in messages:
        parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>")
    if add_generation_prompt:
        parts.append('<|im_start|>assistant\n')
    return '\n'.join(parts)

def chat_text(messages):
    return to_chatml(messages, add_generation_prompt=False)

def make_tool_call(query):
    return '<tool_call>' + json.dumps({'name':'web_search','arguments':{'query':query}}, ensure_ascii=False) + '</tool_call>'

def make_source_block(sources):
    return '\n\n'.join([f"[{s['id']}] {s['title']}\nURL: {s.get('url','local-dataset')}\n{s['text']}" for s in sources])

In [ ]:
seed_examples = [
    {
        'user': 'Is a local 4B model enough to beat GPT-class models at research?',
        'query': 'small language models search tools research assistant accuracy',
        'sources': [
            {'id':'S1','title':'Research-agent design note','url':'local-seed://research-agent','text':'Small models can perform well in narrow workflows when paired with retrieval, tools, verification, and citations. They are weaker than frontier models as raw general reasoning systems.'},
            {'id':'S2','title':'RAG reliability note','url':'local-seed://rag','text':'Retrieval-augmented systems reduce hallucination risk when final answers are constrained to retrieved evidence and uncertainty is stated when evidence is insufficient.'},
        ],
        'answer': 'A 4B model is unlikely to beat frontier models as a raw general model. It can still be useful as a research system if it searches, verifies, and answers only from retrieved evidence [S1][S2].'
    },
    {
        'user': 'Should I answer current benchmark questions from memory?',
        'query': 'latest AI benchmark leaderboard current model scores',
        'sources': [{'id':'S1','title':'Benchmark freshness note','url':'local-seed://benchmarks','text':'Current model benchmark results change frequently. Answers about latest scores should be verified against current leaderboards or official model pages.'}],
        'answer': 'No. Current benchmark questions should trigger search first because leaderboard positions and model scores can change quickly [S1].'
    },
]

def seed_to_rows(examples):
    out = []
    for ex in examples:
        messages = [
            {'role':'system','content':SYSTEM_PROMPT},
            {'role':'user','content':ex['user']},
            {'role':'assistant','content':make_tool_call(ex['query'])},
            {'role':'tool','content':make_source_block(ex['sources'])},
            {'role':'assistant','content':ex['answer']},
        ]
        out.append({'text': chat_text(messages), 'source':'seed_tool_use'})
    return out

rows = seed_to_rows(seed_examples)
print(rows[0]['text'][:1000])

In [ ]:
def build_hotpot_rows(n=500):
    hotpot = load_dataset('hotpot_qa', 'distractor', split=f'train[:{n}]')
    out = []
    for ex in hotpot:
        question, answer = ex['question'], ex['answer']
        titles, sentences_list = ex['context']['title'], ex['context']['sentences']
        sources = []
        for i, (title, sentences) in enumerate(zip(titles[:6], sentences_list[:6])):
            text = ' '.join(sentences[:4]).replace('\n', ' ').strip()
            if text:
                sources.append({'id':f'S{i+1}','title':title,'url':'hotpotqa://wikipedia-context','text':text[:1200]})
        if not sources:
            continue
        citation_ids = []
        try:
            supporting_titles = set(ex['supporting_facts']['title'])
            citation_ids = [f"[{s['id']}]" for s in sources if s['title'] in supporting_titles]
        except Exception:
            pass
        if not citation_ids:
            citation_ids = ['[S1]']
        cited_answer = f"Based on the provided sources, the answer is: {answer}. Evidence: {''.join(citation_ids[:3])}"
        messages = [
            {'role':'system','content':SYSTEM_PROMPT},
            {'role':'user','content':question},
            {'role':'assistant','content':make_tool_call(question)},
            {'role':'tool','content':make_source_block(sources)},
            {'role':'assistant','content':cited_answer},
        ]
        out.append({'text': chat_text(messages), 'source':'hotpot_qa'})
    return out

hotpot_rows = build_hotpot_rows(TRAIN_HOTPOT_SAMPLES)
rows.extend(hotpot_rows)
print('Loaded HotpotQA rows:', len(hotpot_rows))
print('Total rows:', len(rows))

In [ ]:
random.seed(SEED)
random.shuffle(rows)
dataset = Dataset.from_list(rows).shuffle(seed=SEED)
split = dataset.train_test_split(test_size=EVAL_SIZE, seed=SEED)
train_dataset, eval_dataset = split['train'], split['test']
print(train_dataset)
print('Eval:', eval_dataset)
print(train_dataset[0]['text'][:2000])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=text_tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir=f'outputs_{RUN_NAME}',
        optim='adamw_8bit',
        seed=SEED,
        dataset_num_proc=1,
        fp16=True,
        bf16=False,
        report_to='none',
    ),
)
train_result = trainer.train()
print(train_result)

In [ ]:
RUN_NAME = f'{PROJECT_NAME}-{MODEL_SLUG}-lora-step{MAX_STEPS}'
ADAPTER_DIR = f'adapters/{RUN_NAME}'
REPORT_DIR = 'reports'
BUNDLE_DIR = f'runs/{RUN_NAME}'
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(BUNDLE_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
text_tokenizer.save_pretrained(ADAPTER_DIR)

report = {
    'run_name': RUN_NAME,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'base_model': BASE_MODEL,
    'max_steps': MAX_STEPS,
    'max_seq_length': MAX_SEQ_LENGTH,
    'train_hotpot_samples': TRAIN_HOTPOT_SAMPLES,
    'eval_size': EVAL_SIZE,
    'load_in_4bit': LOAD_IN_4BIT,
    'load_in_16bit': LOAD_IN_16BIT,
    'adapter_dir': ADAPTER_DIR,
    'hf_adapter_repo': HF_ADAPTER_REPO,
}
REPORT_PATH = f'{REPORT_DIR}/{RUN_NAME}.json'
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

shutil.copytree(ADAPTER_DIR, f'{BUNDLE_DIR}/adapter', dirs_exist_ok=True)
shutil.copy(REPORT_PATH, f'{BUNDLE_DIR}/report.json')
BUNDLE_ZIP = shutil.make_archive(BUNDLE_DIR, 'zip', BUNDLE_DIR)
print('Saved adapter:', ADAPTER_DIR)
print('Saved report:', REPORT_PATH)
print('Created bundle:', BUNDLE_ZIP)
!ls -lh {BUNDLE_ZIP}

In [ ]:
api = HfApi()
api.create_repo(repo_id=HF_ADAPTER_REPO, repo_type='model', private=True, exist_ok=True)
api.upload_file(
    path_or_fileobj=BUNDLE_ZIP,
    path_in_repo=os.path.basename(BUNDLE_ZIP),
    repo_id=HF_ADAPTER_REPO,
    repo_type='model',
)
print('Uploaded adapter bundle to HF:')
print(f'{HF_ADAPTER_REPO}/{os.path.basename(BUNDLE_ZIP)}')

In [ ]:
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

def generate_text(messages, max_new_tokens=512, temperature=0.2):
    prompt = to_chatml(messages, add_generation_prompt=True)
    inputs = text_tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            pad_token_id=text_tokenizer.eos_token_id,
            eos_token_id=text_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    return text_tokenizer.decode(new_tokens, skip_special_tokens=True)

test_messages = [
    {'role':'system','content':SYSTEM_PROMPT},
    {'role':'user','content':'Compare Qwythos 9B with Gemini Flash for research tasks. Be accurate.'},
]
print(generate_text(test_messages, max_new_tokens=350))

In [ ]:
source_block = '''[S1] Qwythos model card
URL: https://huggingface.co/example/qwythos
Qwythos is a 9B model based on Qwen and advertises long-context capability. Its benchmark claims are self-reported on the model card.

[S2] Public benchmark leaderboard
URL: https://example.com/leaderboard
The public leaderboard does not list Qwythos. Frontier cloud models have independent agent benchmark scores that are much higher than small local models.
'''
test_messages = [
    {'role':'system','content':SYSTEM_PROMPT},
    {'role':'user','content':'Is Qwythos proven to beat Gemini Flash?'},
    {'role':'tool','content':source_block},
]
print(generate_text(test_messages, max_new_tokens=400, temperature=0.2))

## Optional GGUF export

Do not run until adapter output looks useful. Later local download:

```powershell
hf download amaansyed27/kenprobe-gguf <file>.gguf --local-dir models/gguf
```


In [ ]:
DO_GGUF_EXPORT = False
if DO_GGUF_EXPORT:
    GGUF_DIR = f'models/{RUN_NAME}-Q4_K_M'
    model.save_pretrained_gguf(GGUF_DIR, text_tokenizer, quantization_method='q4_k_m')
    api = HfApi()
    api.create_repo(repo_id=HF_GGUF_REPO, repo_type='model', private=True, exist_ok=True)
    for root, _, files in os.walk(GGUF_DIR):
        for name in files:
            if name.endswith('.gguf'):
                local_path = os.path.join(root, name)
                api.upload_file(path_or_fileobj=local_path, path_in_repo=name, repo_id=HF_GGUF_REPO, repo_type='model')
                print('Uploaded:', name)

## Local artifact layout

```powershell
hf auth login
hf download amaansyed27/kenprobe-adapters kenprobe-qwen35-4b-lora-step1000.zip --local-dir runs
mkdir adapters,reports,models,runs -Force
Expand-Archive .\runs\kenprobe-qwen35-4b-lora-step1000.zip -DestinationPath .\runs\step1000 -Force
Copy-Item .\runs\step1000\adapter .\adapters\kenprobe-qwen35-4b-lora-step1000 -Recurse -Force
Copy-Item .\runs\step1000\report.json .\reports\kenprobe-qwen35-4b-lora-step1000.json -Force
```
